# AIMS4PT Calculator

This notebook provides a user-facing calculator for the AIMS4PT framework: AI-assisted Model Selection for Pressure-Temperature estimation. It selects suitable clinopyroxene-based thermobarometers for a given dataset, exports a report summarizing the results.

Before running the calculator, install the AIMS4PT package following the instructions in `README.md`.

Cells marked as `Input required` contain settings that should be checked before execution. In most cases, you only need to edit `input.xlsx`, the optional melt TAS fields, and the project name. Results are written to `report_output/<project_name>/`, and intermediate workflow objects are cached as `.pkl` files so repeated runs can reuse previous calculations.


In [1]:
import pandas as pd
import os
import time
import pickle as pkl
from aims4pt.reporting.excel import report_excel
from aims4pt.model_tools.CpxTBSelect import workflow_thermobarometry

timestamp = time.strftime("%Y%m%d-%H%M%S")
timestamp

'20260514-133147'

# Step 1: Load input data and define basic settings

Enter your sample data in `input.xlsx`, then run the following cells to load clinopyroxene and liquid compositions into the notebook.

The input file should use the column names expected by AIMS4PT. Clinopyroxene oxide columns should end with `_cpx`, and liquid oxide columns should end with `_liq`. Compositions should be reported in wt%.


In [2]:
input_path = 'input.xlsx'
input_data = pd.read_excel(input_path, skiprows=1)  # Skip the first header row.

# Extract clinopyroxene and liquid composition columns from the input table.
X_cpx = input_data.loc[:, input_data.columns.str.endswith('_cpx')]
X_liq = input_data.loc[:, input_data.columns.str.endswith('_liq')]


In [3]:
X_cpx.head()

,SiO2_cpx,TiO2_cpx,Al2O3_cpx,FeOt_cpx,MnO_cpx,MgO_cpx,CaO_cpx,Na2O_cpx,K2O_cpx,Cr2O3_cpx
0,52.06,0.83,3.98,7.72,0.22,15.44,20.33,0.35,0.06,NaN
1,50.74,0.81,3.47,10.20,0.21,16.28,18.12,0.29,0.02,NaN
2,50.45,0.94,2.75,11.38,0.37,15.88,17.17,0.30,0.06,NaN
3,52.10,0.51,2.57,6.69,0.23,16.52,20.39,0.27,0.04,NaN
4,51.44,0.76,3.57,9.29,0.26,16.72,17.70,0.25,0.05,NaN


In [4]:
# Automatically detect whether H2O_liq is empty.
X_water_empty = X_liq["H2O_liq"].isnull().all() 
# Automatically detect whether the liquid composition block is empty.
X_liq_empty = X_liq["SiO2_liq"].isnull().all()  # SiO2_liq is used as the key liquid column.

# Use 0 wt% H2O as a placeholder when H2O_liq is not provided.
if X_water_empty:
    X_liq["H2O_liq"] = 0.0
X_liq.head()


,SiO2_liq,TiO2_liq,Al2O3_liq,FeOt_liq,MnO_liq,MgO_liq,CaO_liq,Na2O_liq,K2O_liq,Cr2O3_liq,P2O5_liq,H2O_liq
0,59.97,0.87,17.37,6.27,0.13,2.89,6.30,4.03,1.90,NaN,0.27,0
1,61.36,0.96,16.51,6.26,0.12,2.46,5.62,4.11,2.25,NaN,0.34,0
2,63.91,0.95,16.28,5.22,0.09,1.63,4.22,4.50,2.78,NaN,0.40,0
3,60.19,0.88,16.84,6.08,0.14,3.42,6.71,3.89,1.71,NaN,0.13,0
4,60.10,0.94,17.66,5.94,0.13,2.91,6.25,4.10,1.82,NaN,0.16,0


## 1.1. Input required, optional: melt TAS fields

If liquid compositions are not available, provide the assumed melt TAS fields for the melt-composition OOD check. These fields help the framework evaluate whether each thermobarometer is applicable to the inferred melt composition.

Valid melt TAS fields are:

```python
VALID_MELT_TAS_FIELDS = (
    "Picrite",
    "Basalt",
    "Basaltic Andesite",
    "Andesite",
    "Dacite",
    "Rhyolite",
    "Foidite",
    "Trachyte",
    "Trachybasalt",
    "Basaltic Trachyandesite",
    "Trachyandesite",
    "Tephrite/Basanite",
    "Phonotephrite",
    "Tephriphonolite",
    "Phonolite",
)
```

Edit `TAS_fields` as a Python list, for example `TAS_fields = ["Basalt", "Andesite"]`.


In [5]:
# Default melt TAS fields used when liquid compositions are unavailable.
TAS_fields = ['Basalt', 'Andesite', ]  # Edit this list to match the assumed melt fields for your dataset.



## 1.2. Input required: basic AIMS4PT settings

Set whether clinopyroxene-liquid thermobarometers should be evaluated and choose a project name for the output folder and report files.


In [6]:
# Set whether to calculate pressure-temperature estimates from clinopyroxene-liquid thermobarometers.
calculate_cpx_liq = True

# Project name used for output file names and result folders.
project_name = "cpx_liq_example_project"  # Edit this value for your project.


# Step 2: Run the AIMS4PT calculator

Run the following cells to initialize the thermobarometer pools, evaluate model applicability, calculate pressure-temperature estimates, and export Excel reports to `report_output/<project_name>/`.


In [7]:
# Create the output directory if it does not already exist.
output_dir = os.path.join("report_output", project_name)
os.makedirs(output_dir, exist_ok=True)


## 2.1. Initialize thermobarometer model pools


In [8]:
from aims4pt.model_tools.model_registry import  get_models_initial_pools, ALL_MODELS_MODULES


import importlib
for module in ALL_MODELS_MODULES:
    importlib.import_module(module)

if X_liq_empty:
    P_model_pool = get_models_initial_pools("P", "cpx_only" , False) 
    T_model_pool = get_models_initial_pools("T", "cpx_only" , False)
    print("No melt composition data detected. Only clinopyroxene-based models will be calculated.")
else:    
    P_model_pool = get_models_initial_pools("P", "both" , False) 
    T_model_pool = get_models_initial_pools("T", "both" , False)
    print("Melt composition data detected. Both clinopyroxene-based and clinopyroxene-liquid-based models will be calculated.")

T_model_names = list(model.model_name for model in T_model_pool)
P_model_names = list(model.model_name for model in P_model_pool)
# Exclude selected models here if needed, for example:
# models_to_exclude = ["Brugman & Till, 2019 (cpx_liq)", ]  # Example model names to exclude.
# P_model_pool = [model for model in P_model_pool if model.model_name not in models_to_exclude]
# T_model_pool = [model for model in T_model_pool if model.model_name not in models_to_exclude]
# P_model_names = list(model.model_name for model in P_model_pool)
# T_model_names = list(model.model_name for model in T_model_pool)

[Skip] Error initializing model Brugman_Till_19: Brugman_Till_19 only supports temperature (T) calculations, not pressure (P) calculations.


c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.7.2 when using ver

[Skip] Error initializing model Brugman_Till_19: Brugman_Till_19 only supports temperature (T) calculations, not pressure (P) calculations.


c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.7.2 when using ver

[skipped] Error initializing model <class 'aims4pt.model_tools.Neave_Putirka_17.Neave_Putirka_17'> with T: Pu08_eq33_T, P: eq1_P - Neave_Putirka_17 only supports pressure (P) calculations, not temperature (T) calculations.


c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Melt composition data detected. Both clinopyroxene-based and clinopyroxene-liquid-based models will be calculated.


c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.7.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\miniconda3\envs\AIMS4PT\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1

In [9]:
# Display all pressure models in the pool.
# Check model names here before excluding selected models.
P_model_pool


[
         ModelManager: Putirka_08
         Model name: Putirka, 2008 eq32d_T; eq32a_P (cpx_only)
         interested parameter: P; cpx-only
         require_water: False
         uncertainty: 3.1 kbar
         comment: None
         ,
 
         ModelManager: Putirka_08
         Model name: Putirka, 2008 eq32d_T; eq32b_P (cpx_only)
         interested parameter: P; cpx-only
         require_water: True
         uncertainty: 2.6 kbar
         comment: None
         ,
 
         ModelManager: Petrelli20
         Model name: Petrelli et al., 2020 (cpx_only)
         interested parameter: P; cpx-only
         require_water: False
         uncertainty: 3.2 kbar
         comment: None
         ,
 
         ModelManager: Higgins21
         Model name: Higgins et al., 2021 (cpx_only)
         interested parameter: P; cpx-only
         require_water: False
         uncertainty: 2.3 kbar
         comment: None
         ,
 
         ModelManager: Jorgenson22
         Model name: Jorgenson et al

In [10]:
T_model_pool

[
         ModelManager: Putirka_08
         Model name: Putirka, 2008 eq32d_T; eq32a_P (cpx_only)
         interested parameter: T; cpx-only
         require_water: False
         uncertainty: 58 ℃
         comment: None
         ,
 
         ModelManager: Putirka_08
         Model name: Putirka, 2008 eq32d_T; eq32b_P (cpx_only)
         interested parameter: T; cpx-only
         require_water: True
         uncertainty: 58 ℃
         comment: None
         ,
 
         ModelManager: Petrelli20
         Model name: Petrelli et al., 2020 (cpx_only)
         interested parameter: T; cpx-only
         require_water: False
         uncertainty: 96 ℃
         comment: None
         ,
 
         ModelManager: Higgins21
         Model name: Higgins et al., 2021 (cpx_only)
         interested parameter: T; cpx-only
         require_water: False
         uncertainty: 57 ℃
         comment: None
         ,
 
         ModelManager: Jorgenson22
         Model name: Jorgenson et al., 2022 (cpx_onl

## 2.2. Clinopyroxene-only barometry and thermometry


### 2.2.1. Barometers


In [11]:

P_cpx_only_pool = [model for model in P_model_pool if model.cpx_only]

cache_path_P_only = os.path.join(output_dir, f"computed_cache_cpx_only_P.pkl")


if os.path.exists(cache_path_P_only):
    with open(cache_path_P_only, 'rb') as f:
        framework_P_cpx_only = pkl.load(f)
    print("Loaded existing workflow from:", cache_path_P_only)
else:
    framework_P_cpx_only = workflow_thermobarometry(P_cpx_only_pool)
    if X_liq_empty:
        # If liquid data are unavailable, use the assumed melt TAS fields for OOD checking.
        framework_P_cpx_only.predict(X_cpx, X_liq, input_melt_TAS=TAS_fields, melt_TAS_source = "input_melt_TAS")
    else:
        # Otherwise, use the measured liquid compositions for OOD checking.
        framework_P_cpx_only.predict(X_cpx, X_liq)
    with open(cache_path_P_only, 'wb') as f:
        pkl.dump(framework_P_cpx_only, f)
    print("Saved workflow to:", cache_path_P_only)




report_path_P_only = os.path.join(output_dir, f"{timestamp}_report_cpx_only_P.xlsx")

report_excel(
    workflow_obj=framework_P_cpx_only,
    model_list =P_cpx_only_pool,
    original_data=input_data,
    out_path=report_path_P_only
)


Loaded existing workflow from: report_output\cpx_liq_example_project\computed_cache_cpx_only_P.pkl


### 2.2.2. Thermometers


In [12]:

T_cpx_only_pool = [model for model in T_model_pool if model.cpx_only]

cache_path_T_only = os.path.join(output_dir, f"computed_cache_cpx_only_T.pkl")


if os.path.exists(cache_path_T_only):
    with open(cache_path_T_only, 'rb') as f:
        framework_T_cpx_only = pkl.load(f)
    print("Loaded existing workflow from:", cache_path_T_only)
else:
    framework_T_cpx_only = workflow_thermobarometry(T_cpx_only_pool)
    if X_liq_empty:
        # If liquid data are unavailable, use the assumed melt TAS fields for OOD checking.
        framework_T_cpx_only.predict(X_cpx, X_liq, input_melt_TAS=TAS_fields, melt_TAS_source = "input_melt_TAS")
    else:
        # Otherwise, use the measured liquid compositions for OOD checking.
        framework_T_cpx_only.predict(X_cpx, X_liq)
    with open(cache_path_T_only, 'wb') as f:
        pkl.dump(framework_T_cpx_only, f)
    print("Saved workflow to:", cache_path_T_only)




report_path_T_only = os.path.join(output_dir, f"{timestamp}_report_cpx_only_T.xlsx")

report_excel(
    workflow_obj=framework_T_cpx_only,
    model_list =T_cpx_only_pool,
    original_data=input_data,
    out_path=report_path_T_only
)


Loaded existing workflow from: report_output\cpx_liq_example_project\computed_cache_cpx_only_T.pkl


## 2.3. Clinopyroxene-liquid barometry and thermometry

Run this section only when liquid compositions are provided in `input.xlsx`.


In [13]:
# Stop this section when no liquid compositions are detected.
if X_liq_empty:
    raise ValueError("No liquid data detected. Clinopyroxene-liquid models will not be calculated.")
elif not calculate_cpx_liq:
    raise ValueError("Clinopyroxene-liquid calculation is disabled. Set calculate_cpx_liq = True to enable these calculations.")

### 2.3.1. Barometers


In [ ]:



P_cpx_liq_pool = [model for model in P_model_pool if not model.cpx_only]

cache_path_P_liq = os.path.join(output_dir, f"computed_cache_cpx_liq_P.pkl")


if os.path.exists(cache_path_P_liq):
    with open(cache_path_P_liq, 'rb') as f:
        framework_P_cpx_liq = pkl.load(f)
    print("Loaded existing workflow from:", cache_path_P_liq)
else:
    framework_P_cpx_liq = workflow_thermobarometry(P_cpx_liq_pool)
    framework_P_cpx_liq.predict(X_cpx, X_liq)
    with open(cache_path_P_liq, 'wb') as f:
        pkl.dump(framework_P_cpx_liq, f)
    print("Saved workflow to:", cache_path_P_liq)




report_path_P_liq = os.path.join(output_dir, f"{timestamp}_report_cpx_liq_P.xlsx")

report_excel(
    workflow_obj=framework_P_cpx_liq,
    model_list =P_cpx_liq_pool,
    original_data=input_data,
    out_path=report_path_P_liq
)


Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.
R callback write-console: Loading required package: PerformanceAnalytics
  
R callback write-console: Loading required package: xts
  
R callback write-console: Loading required package: zoo
  
R callback write-console: 
Attaching package: 'zoo'

  
R callback write-console: The following objects are masked from 'package:base':

    as.Date, as.Date.numeric

  
R callback write-console: 
Attaching package: 'PerformanceAnalytics'

  
R callback write-console: The following object is masked from 'package:graphics':

    legend

  
R callback write-console: Loading required package: rJava
  
R callback write-console: Loading required package: extraTrees
  
R callback write-console: Loading required package: readxl
  
R callback write-console: Loading required package: EnvStats
  
R callback write-console: 
Attaching package: 'EnvStats'

  
R callback write-console: The foll


Saved workflow to: report_output\cpx_liq_example_project\——computed_cache_cpx_liq_P.pkl


### 2.3.2. Thermometers


In [15]:

T_cpx_liq_pool = [model for model in T_model_pool if not model.cpx_only]

cache_path_T_liq = os.path.join(output_dir, f"computed_cache_cpx_liq_T.pkl")


if os.path.exists(cache_path_T_liq):
    with open(cache_path_T_liq, 'rb') as f:
        framework_T_cpx_liq = pkl.load(f)
    print("Loaded existing workflow from:", cache_path_T_liq)
else:
    framework_T_cpx_liq = workflow_thermobarometry(T_cpx_liq_pool)
    framework_T_cpx_liq.predict(X_cpx, X_liq)
    with open(cache_path_T_liq, 'wb') as f:
        pkl.dump(framework_T_cpx_liq, f)
    print("Saved workflow to:", cache_path_T_liq)




report_path_T_liq = os.path.join(output_dir, f"{timestamp}_report_cpx_liq_T.xlsx")

report_excel(
    workflow_obj=framework_T_cpx_liq,
    model_list =T_cpx_liq_pool,
    original_data=input_data,
    out_path=report_path_T_liq
)


Loaded existing workflow from: report_output\cpx_liq_example_project\computed_cache_cpx_liq_T.pkl
